# Quick Start: Running Chronos-2 on BOOM benchmark

This notebook is adapted from the [GiftEval repository](https://github.com/SalesforceAIResearch/gift-eval/tree/main/notebooks) and shows how to run [Chronos-2](https://huggingface.co/amazon/chronos-2) on the BOOM benchmark.

Chronos-2 is a 120M-parameter encoder-only time series foundation model that supports univariate, multivariate, and covariate-informed forecasting. It produces multi-step-ahead quantile forecasts and supports cross-series in-context learning via `predict_batches_jointly=True`.

`context_length` is set to 2048 for all evaluations in this notebook.

Make sure you download the BOOM benchmark and set the `BOOM` environment variable correctly before running this notebook.

We will use the `Dataset` class from GiftEval to load the data and run the model.

Download BOOM datasets. Calling `download_boom_benchmark` also sets the `BOOM` environment variable with the correct path, which is needed for running the evals below.

In [ ]:
import os
import json
from dotenv import load_dotenv
from dataset_utils import download_boom_benchmark

boom_path = "ChangeMe"
download_boom_benchmark(boom_path)
load_dotenv()

dataset_properties_map = json.load(open("./boom/boom_properties.json"))
all_datasets = list(dataset_properties_map.keys())
print(len(all_datasets))

In [ ]:
from gluonts.ev.metrics import (
    MAE,
    MAPE,
    MASE,
    MSE,
    MSIS,
    ND,
    NRMSE,
    RMSE,
    SMAPE,
    MeanWeightedSumQuantileLoss,
)

# Instantiate the metrics
metrics = [
    MSE(forecast_type="mean"),
    MSE(forecast_type=0.5),
    MAE(),
    MASE(),
    MAPE(),
    SMAPE(),
    MSIS(),
    RMSE(),
    NRMSE(),
    ND(),
    MeanWeightedSumQuantileLoss(quantile_levels=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]),
]

## Chronos-2 Predictor

We wrap `Chronos2Pipeline.predict_quantiles()` in a gluonts-compatible predictor. `context_length=2048` is forwarded to the pipeline, which truncates inputs to the last 2048 timesteps.

In [ ]:
import logging
from typing import List

import numpy as np
import torch
from chronos import BaseChronosPipeline, Chronos2Pipeline
from gluonts.model import Forecast
from gluonts.model.forecast import QuantileForecast

logger = logging.getLogger("Chronos-2 Predictor")
logger.setLevel(logging.INFO)

QUANTILE_LEVELS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
CONTEXT_LENGTH = 2048


class Chronos2Predictor:
    def __init__(
        self,
        model_path: str,
        prediction_length: int,
        batch_size: int = 100,
        quantile_levels: List[float] = QUANTILE_LEVELS,
        predict_batches_jointly: bool = True,
        context_length: int = CONTEXT_LENGTH,
        **kwargs,
    ):
        self.pipeline = BaseChronosPipeline.from_pretrained(model_path, **kwargs)
        assert isinstance(self.pipeline, Chronos2Pipeline), (
            "This predictor is for Chronos-2. See the Chronos notebook for Chronos / Chronos-Bolt."
        )
        self.prediction_length = prediction_length
        self.batch_size = batch_size
        self.quantile_levels = quantile_levels
        self.predict_batches_jointly = predict_batches_jointly
        self.context_length = context_length

    def predict(self, test_data_input) -> List[Forecast]:
        input_data = [{"target": item["target"]} for item in test_data_input]
        is_univariate = input_data[0]["target"].ndim == 1

        model_batch_size = self.batch_size
        while True:
            try:
                quantiles, _ = self.pipeline.predict_quantiles(
                    inputs=input_data,
                    prediction_length=self.prediction_length,
                    batch_size=model_batch_size,
                    quantile_levels=self.quantile_levels,
                    predict_batches_jointly=self.predict_batches_jointly,
                    context_length=self.context_length,
                )
                # quantiles: list of (n_variates, pred_len, n_quantiles)
                arr = torch.stack(quantiles).permute(0, 3, 2, 1).cpu().numpy()
                if is_univariate:
                    arr = arr.squeeze(-1)
                break
            except torch.cuda.OutOfMemoryError:
                logger.error(f"OOM at batch_size {model_batch_size}, halving")
                model_batch_size //= 2

        forecasts = []
        for preds, ts in zip(arr, test_data_input):
            start_date = ts["start"] + len(ts["target"])
            forecasts.append(
                QuantileForecast(
                    forecast_arrays=preds,
                    forecast_keys=list(map(str, self.quantile_levels)),
                    start_date=start_date,
                )
            )
        return forecasts

## Evaluation

We iterate over all BOOM datasets and terms, run the predictor, and record metrics to `results/{model_name}/all_results.csv`.

Because `predict_batches_jointly=True` enables cross-series in-context learning, we predict each rolling window separately to avoid leakage across windows of the same series.

In [ ]:
import csv
import itertools
import os
import logging

from gluonts.model import evaluate_forecasts
from gluonts.time_feature import get_seasonality
from gift_eval.data import Dataset

# Suppress the gluonts warning about missing mean predictions
class _Filter(logging.Filter):
    def filter(self, record):
        return "The mean prediction is not stored in the forecast data" not in record.getMessage()
logging.getLogger("gluonts.model.forecast").addFilter(_Filter())

model_name = "chronos_2"
output_dir = f"ChangeMe/{model_name}"
os.makedirs(output_dir, exist_ok=True)
csv_file_path = os.path.join(output_dir, "all_results.csv")

with open(csv_file_path, "w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(
        [
            "dataset",
            "model",
            "eval_metrics/MSE[mean]",
            "eval_metrics/MSE[0.5]",
            "eval_metrics/MAE[0.5]",
            "eval_metrics/MASE[0.5]",
            "eval_metrics/MAPE[0.5]",
            "eval_metrics/sMAPE[0.5]",
            "eval_metrics/MSIS",
            "eval_metrics/RMSE[mean]",
            "eval_metrics/NRMSE[mean]",
            "eval_metrics/ND[0.5]",
            "eval_metrics/mean_weighted_sum_quantile_loss",
            "domain",
            "num_variates",
            "dataset_size",
        ]
    )

for ds_num, ds_name in enumerate(all_datasets):
    print(f"Processing dataset: {ds_name} ({ds_num + 1} of {len(all_datasets)})")
    dataset_term = dataset_properties_map[ds_name]["term"]
    for term in ["short", "medium", "long"]:
        if (term == "medium" or term == "long") and dataset_term == "short":
            continue

        ds_freq = dataset_properties_map[ds_name]["frequency"]
        ds_config = f"{ds_name}/{ds_freq}/{term}"

        # Keep multivariate data multivariate so Chronos-2 jointly forecasts all variates
        dataset = Dataset(name=ds_name, term=term, to_univariate=False, storage_env_var="BOOM")

        season_length = get_seasonality(dataset.freq)
        dataset_size = len(dataset.test_data)
        print(f"Dataset size: {dataset_size}")

        predictor = Chronos2Predictor(
            model_path="amazon/chronos-2",
            prediction_length=dataset.prediction_length,
            batch_size=100,
            quantile_levels=QUANTILE_LEVELS,
            predict_batches_jointly=True,
            context_length=CONTEXT_LENGTH,
            device_map="cuda:0",
            torch_dtype="float32",
        )

        # Predict each rolling window separately to avoid cross-window leakage
        n_windows = dataset.test_data.windows
        forecast_windows = []
        for window_idx in range(n_windows):
            entries_window_k = list(
                itertools.islice(dataset.test_data.input, window_idx, None, n_windows)
            )
            forecast_windows.append(list(predictor.predict(entries_window_k)))
        forecasts = [item for items in zip(*forecast_windows) for item in items]

        res = evaluate_forecasts(
            forecasts,
            test_data=dataset.test_data,
            metrics=metrics,
            batch_size=1024,
            axis=None,
            mask_invalid_label=True,
            allow_nan_forecast=False,
            seasonality=season_length,
        )

        with open(csv_file_path, "a", newline="") as csvfile:
            writer = csv.writer(csvfile)
            writer.writerow(
                [
                    ds_config,
                    model_name,
                    res["MSE[mean]"].iloc[0],
                    res["MSE[0.5]"].iloc[0],
                    res["MAE[0.5]"].iloc[0],
                    res["MASE[0.5]"].iloc[0],
                    res["MAPE[0.5]"].iloc[0],
                    res["sMAPE[0.5]"].iloc[0],
                    res["MSIS"].iloc[0],
                    res["RMSE[mean]"].iloc[0],
                    res["NRMSE[mean]"].iloc[0],
                    res["ND[0.5]"].iloc[0],
                    res["mean_weighted_sum_quantile_loss"].iloc[0],
                    dataset_properties_map[ds_name]["domain"],
                    dataset_properties_map[ds_name]["num_variates"],
                    dataset_size,
                ]
            )
        print(f"Results for {ds_name} have been written to {csv_file_path}")

## Results

Running the above cell will generate `all_results.csv` under `results/chronos_2/`. We can display it:

In [ ]:
import pandas as pd
df = pd.read_csv(output_dir + "/all_results.csv")
df